# Claude Architect Bootcamp — Week 1, Session 1
## Claude API Beyond Chat

Model: `claude-sonnet-4-6` | SDK: `anthropic` | No frameworks.

> **Restart & Clear Output before going live.**
>
> Kernel → Restart & Clear Output. Stale outputs from rehearsal will confuse the room.

In [1]:
# Setup — install, import, helpers
%pip install -q anthropic python-dotenv

import ast
import json
import operator
import os
import time
from typing import Any

from anthropic import Anthropic, APIStatusError
from dotenv import load_dotenv
from IPython.display import HTML, display

load_dotenv()

MODEL = "claude-sonnet-4-6"
client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])


def show_response(resp) -> None:
    """Pretty panel: content blocks, stop_reason, usage."""
    print("=" * 72)
    print("CONTENT BLOCKS")
    print("=" * 72)
    for i, block in enumerate(resp.content):
        block_type = getattr(block, "type", type(block).__name__)
        print(f"\n--- Block {i + 1}: type={block_type} ---")
        if block_type == "text":
            print(block.text)
        elif block_type == "tool_use":
            print(f"tool name: {block.name}")
            print("tool input:")
            print(json.dumps(block.input, indent=2))
        else:
            print(json.dumps(block.model_dump() if hasattr(block, "model_dump") else str(block), indent=2))

    print("\n" + "=" * 72)
    display(HTML(f"<h3 style='color:#c2410c;margin:8px 0;'>stop_reason: <b>{resp.stop_reason}</b></h3>"))

    usage = resp.usage.model_dump()
    print("USAGE")
    print(json.dumps(usage, indent=2))
    print("=" * 72)


def call_with_retry(**kwargs):
    """Retry once on 429/529 — never crash mid-demo."""
    try:
        return client.messages.create(**kwargs)
    except APIStatusError as e:
        if e.status_code in (429, 529):
            print("Rate limited — retrying in 2s. This is why production code always has retries.")
            time.sleep(2)
            return client.messages.create(**kwargs)
        raise

print("Ready. Model:", MODEL)


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Ready. Model: claude-sonnet-4-6


## Section 1 — First call + reading the raw response

### The API returns a structured object, not just text

**Claim:** A normal `messages.create` call returns content blocks, a stop reason, and token usage.

In [2]:
resp1 = call_with_retry(
    model=MODEL,
    max_tokens=1024,
    system="You are a concise assistant for a financial services company.",
    messages=[{"role": "user", "content": "What is prompt caching in one paragraph?"}],
)
show_response(resp1)

CONTENT BLOCKS

--- Block 1: type=text ---
Prompt caching is a technique used in large language model (LLM) systems where frequently used or repeated portions of input prompts are stored in a cache so they don't need to be reprocessed from scratch with every API call. When a prompt contains a long, repeated prefix — such as a system instruction, a large document, or a set of examples — the model can retrieve the cached computation (typically the key-value attention states) instead of recomputing it, significantly reducing latency and cost. This is particularly useful in applications where the same context or instructions are sent repeatedly across many requests, such as chatbots with long system prompts or document analysis tools, as it can cut processing time and token costs substantially.



USAGE
{
  "cache_creation": {
    "ephemeral_1h_input_tokens": 0,
    "ephemeral_5m_input_tokens": 0
  },
  "cache_creation_input_tokens": 0,
  "cache_read_input_tokens": 0,
  "inference_geo": "global",
  "input_tokens": 29,
  "output_tokens": 154,
  "output_tokens_details": null,
  "server_tool_use": null,
  "service_tier": "standard"
}


**Look at the output above:**
- `content` is a **LIST** of blocks (usually one `text` block today — foreshadows `tool_use` blocks in Section 2)
- `stop_reason` == `"end_turn"` — Claude finished naturally
- `usage` — `input_tokens` and `output_tokens` are your cost and capacity signals

**Claim:** `max_tokens` is a hard ceiling — hit it and the answer truncates mid-sentence.

In [3]:
resp2 = call_with_retry(
    model=MODEL,
    max_tokens=50,
    system="You are a concise assistant for a financial services company.",
    messages=[{"role": "user", "content": "What is prompt caching in one paragraph?"}],
)
show_response(resp2)

CONTENT BLOCKS

--- Block 1: type=text ---
Prompt caching is a technique used in large language model (LLM) systems where frequently used or repeated portions of a prompt — such as system instructions, context documents, or conversation history — are stored in a cache so they don't need



USAGE
{
  "cache_creation": {
    "ephemeral_1h_input_tokens": 0,
    "ephemeral_5m_input_tokens": 0
  },
  "cache_creation_input_tokens": 0,
  "cache_read_input_tokens": 0,
  "inference_geo": "global",
  "input_tokens": 29,
  "output_tokens": 50,
  "output_tokens_details": null,
  "server_tool_use": null,
  "service_tier": "standard"
}


> **In an agent loop, unhandled `max_tokens` = silent data corruption.** Assignment pass criterion #1.

## Section 2 — The tool calling loop, by hand

### Claude never runs code — it asks, your runtime executes

In [4]:
# Tool implementations (demo only — never raw eval() in production)

MOCK_PRICES = {"NVDA": 1284.50, "AAPL": 243.10, "GOOG": 201.75}

_ALLOWED_OPS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Pow: operator.pow,
    ast.USub: operator.neg,
}


def _safe_eval_node(node):
    if isinstance(node, ast.Expression):
        return _safe_eval_node(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _ALLOWED_OPS:
        return _ALLOWED_OPS[type(node.op)](_safe_eval_node(node.left), _safe_eval_node(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _ALLOWED_OPS:
        return _ALLOWED_OPS[type(node.op)](_safe_eval_node(node.operand))
    raise ValueError(f"Unsupported expression: {ast.dump(node)}")


def calculator(expression: str) -> float:
    # never raw eval() in production
    tree = ast.parse(expression.strip(), mode="eval")
    return float(_safe_eval_node(tree))


def get_stock_price(ticker: str) -> dict:
    price = MOCK_PRICES.get(ticker.upper())
    if price is None:
        # structured errors, not generic failures — Week 1 assignment criterion #3
        return {"error": "Unknown ticker", "errorCategory": "validation", "isRetryable": False}
    return {"ticker": ticker.upper(), "price_usd": price}


TOOL_HANDLERS = {
    "calculator": lambda args: calculator(args["expression"]),
    "get_stock_price": lambda args: get_stock_price(args["ticker"]),
}

TOOLS = [
    {
        "name": "calculator",
        "description": (
            "Evaluates a basic arithmetic expression. Use for any math the user asks for. "
            "Input is a single expression string like '(1420 * 3) + 275'. Supports + - * / ** and parentheses. "
            "Do not use for currency conversion or anything requiring live data."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "Arithmetic expression to evaluate, e.g. '23 * 41'",
                }
            },
            "required": ["expression"],
        },
    },
    {
        "name": "get_stock_price",
        "description": (
            "Returns the latest closing price in USD for a stock ticker symbol, e.g. 'NVDA', 'AAPL'. "
            "Use only when the user asks about a specific stock's price. Returns mock data in this demo."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "ticker": {
                    "type": "string",
                    "description": "Stock ticker symbol, uppercase, e.g. 'NVDA'",
                }
            },
            "required": ["ticker"],
        },
    },
]

print("Tools defined:", [t["name"] for t in TOOLS])
print("calculator('23 * 41') =>", calculator("23 * 41"))

Tools defined: ['calculator', 'get_stock_price']
calculator('23 * 41') => 943.0


**Claim:** When tools are available, Claude can respond with `stop_reason: tool_use` instead of text.

In [5]:
single_tool_call = call_with_retry(
    model=MODEL,
    max_tokens=1024,
    tools=TOOLS,
    messages=[{"role": "user", "content": "What is NVIDIA's stock price?"}],
)
show_response(single_tool_call)

for block in single_tool_call.content:
    if block.type == "tool_use":
        print(f"\nClaude chose tool: {block.name}")
        print("Arguments:")
        print(json.dumps(block.input, indent=2))

CONTENT BLOCKS

--- Block 1: type=tool_use ---
tool name: get_stock_price
tool input:
{
  "ticker": "NVDA"
}



USAGE
{
  "cache_creation": {
    "ephemeral_1h_input_tokens": 0,
    "ephemeral_5m_input_tokens": 0
  },
  "cache_creation_input_tokens": 0,
  "cache_read_input_tokens": 0,
  "inference_geo": "global",
  "input_tokens": 760,
  "output_tokens": 58,
  "output_tokens_details": null,
  "server_tool_use": null,
  "service_tier": "standard"
}

Claude chose tool: get_stock_price
Arguments:
{
  "ticker": "NVDA"
}


> **Claude never runs code. It ASKS. Your runtime executes.**

**Claim:** A production agent loop branches on `stop_reason`, not iteration count or string matching.

In [6]:
def _format_tool_result(name: str, result) -> str:
    if name == "get_stock_price" and isinstance(result, dict) and "price_usd" in result:
        return str(result["price_usd"])
    return json.dumps(result) if not isinstance(result, (int, float)) else str(result)


def run_agent_loop(user_message: str, verbose: bool = True) -> str:
    messages = [{"role": "user", "content": user_message}]
    iteration = 0

    while True:
        # ANTI-PATTERN: while i < 10 as your stop condition
        # ANTI-PATTERN: if 'done' in response.text
        iteration += 1
        resp = call_with_retry(model=MODEL, max_tokens=1024, tools=TOOLS, messages=messages)

        if resp.stop_reason == "tool_use":
            tool_results = []
            called = []
            for block in resp.content:
                if block.type == "tool_use":
                    handler = TOOL_HANDLERS[block.name]
                    result = handler(block.input)
                    called.append((block.name, block.input, result))
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": json.dumps(result),
                    })
            if verbose:
                for name, args, result in called:
                    print(
                        f"=== ITERATION {iteration} === stop_reason: tool_use | "
                        f"Claude called: {name}({json.dumps(args)}) | "
                        f"returned: {_format_tool_result(name, result)}"
                    )
            messages.append({"role": "assistant", "content": resp.content})
            messages.append({"role": "user", "content": tool_results})
            continue

        if resp.stop_reason == "end_turn":
            final_text = "".join(b.text for b in resp.content if b.type == "text")
            if verbose:
                print(f"=== ITERATION {iteration} === stop_reason: end_turn | final answer:")
                print(final_text)
            return final_text

        if resp.stop_reason == "max_tokens":
            raise RuntimeError("Unhandled max_tokens — silent data corruption risk")

        raise RuntimeError(f"Unexpected stop_reason: {resp.stop_reason}")


print("Agent loop defined.")


Agent loop defined.


**Claim:** Multi-step questions require multiple tool rounds — price lookup, then math.

In [7]:
run_agent_loop(
    "If I buy 150 shares of NVIDIA at the current price, what will it cost me in total?"
)

=== ITERATION 1 === stop_reason: tool_use | Claude called: get_stock_price({"ticker": "NVDA"}) | returned: 1284.5
=== ITERATION 2 === stop_reason: tool_use | Claude called: calculator({"expression": "150 * 1284.5"}) | returned: 192675.0
=== ITERATION 3 === stop_reason: end_turn | final answer:
Here's a summary of your potential purchase:

- 📈 **NVIDIA (NVDA) Current Price:** $1,284.50 per share
- 🔢 **Number of Shares:** 150
- 💰 **Total Cost: $192,675.00**

Please note that this price is based on the latest available closing price and may not reflect real-time market fluctuations. Additionally, brokerage fees or commissions may add to the total cost. Would you like to know anything else?


"Here's a summary of your potential purchase:\n\n- 📈 **NVIDIA (NVDA) Current Price:** $1,284.50 per share\n- 🔢 **Number of Shares:** 150\n- 💰 **Total Cost: $192,675.00**\n\nPlease note that this price is based on the latest available closing price and may not reflect real-time market fluctuations. Additionally, brokerage fees or commissions may add to the total cost. Would you like to know anything else?"

## Section 3 — Structured output the exam way (`tool_use` + JSON schema)

### A valid schema can still contain wrong data

In [8]:
INVOICE_TEXT = """INVOICE #INV-2026-0847
From: Meridian Consulting Ltd
To: Apex Retail Group
Date: 12 August 2026
Line items:
- Strategy workshop facilitation: $4,500
- Market analysis report: $3,200
- Stakeholder interviews (12 sessions): $2,800
Total due: $11,300
Payment terms: Net 30. A late fee of 2% applies after the due date."""

EXTRACT_INVOICE_TOOL = {
    "name": "extract_invoice",
    "description": "Extract structured fields from an invoice document.",
    "input_schema": {
        "type": "object",
        "properties": {
            "invoice_number": {"type": "string"},
            "vendor": {"type": "string"},
            "client": {"type": "string"},
            "invoice_date": {"type": "string", "description": "ISO 8601 date"},
            "line_items": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "description": {"type": "string"},
                        "amount_usd": {"type": "number"},
                    },
                    "required": ["description", "amount_usd"],
                },
            },
            "stated_total_usd": {"type": "number"},
            "payment_terms": {
                "type": "string",
                "enum": ["net_15", "net_30", "net_60", "due_on_receipt", "other"],
            },
            "payment_terms_detail": {
                "type": "string",
                "description": "Required if payment_terms is 'other'",
            },
            "purchase_order": {
                "type": ["string", "null"],
                "description": "PO number if present, null if absent — DO NOT invent one",
            },
        },
        "required": [
            "invoice_number", "vendor", "client", "invoice_date",
            "line_items", "stated_total_usd", "payment_terms",
        ],
    },
}

print(INVOICE_TEXT)

INVOICE #INV-2026-0847
From: Meridian Consulting Ltd
To: Apex Retail Group
Date: 12 August 2026
Line items:
- Strategy workshop facilitation: $4,500
- Market analysis report: $3,200
- Stakeholder interviews (12 sessions): $2,800
Total due: $11,300
Payment terms: Net 30. A late fee of 2% applies after the due date.


**Claim:** Forced `tool_choice` returns schema-compliant JSON — including correctly null fields.

In [9]:
extract_resp = call_with_retry(
    model=MODEL,
    max_tokens=2048,
    tools=[EXTRACT_INVOICE_TOOL],
    tool_choice={"type": "tool", "name": "extract_invoice"},
    messages=[
        {
            "role": "user",
            "content": f"Extract all invoice fields from this document:\n\n{INVOICE_TEXT}",
        }
    ],
)
show_response(extract_resp)

extracted = next(b.input for b in extract_resp.content if b.type == "tool_use")
print("\nExtracted JSON:")
print(json.dumps(extracted, indent=2))

CONTENT BLOCKS

--- Block 1: type=tool_use ---
tool name: extract_invoice
tool input:
{
  "invoice_number": "INV-2026-0847",
  "vendor": "Meridian Consulting Ltd",
  "client": "Apex Retail Group",
  "invoice_date": "2026-08-12",
  "line_items": [
    {
      "description": "Strategy workshop facilitation",
      "amount_usd": 4500
    },
    {
      "description": "Market analysis report",
      "amount_usd": 3200
    },
    {
      "description": "Stakeholder interviews (12 sessions)",
      "amount_usd": 2800
    }
  ],
  "stated_total_usd": 11300,
  "payment_terms": "net_30"
}



USAGE
{
  "cache_creation": {
    "ephemeral_1h_input_tokens": 0,
    "ephemeral_5m_input_tokens": 0
  },
  "cache_creation_input_tokens": 0,
  "cache_read_input_tokens": 0,
  "inference_geo": "global",
  "input_tokens": 1027,
  "output_tokens": 233,
  "output_tokens_details": null,
  "server_tool_use": null,
  "service_tier": "standard"
}

Extracted JSON:
{
  "invoice_number": "INV-2026-0847",
  "vendor": "Meridian Consulting Ltd",
  "client": "Apex Retail Group",
  "invoice_date": "2026-08-12",
  "line_items": [
    {
      "description": "Strategy workshop facilitation",
      "amount_usd": 4500
    },
    {
      "description": "Market analysis report",
      "amount_usd": 3200
    },
    {
      "description": "Stakeholder interviews (12 sessions)",
      "amount_usd": 2800
    }
  ],
  "stated_total_usd": 11300,
  "payment_terms": "net_30"
}


> **Nullable fields stop the model inventing data** — `purchase_order: null` because no PO appears in the source. Exam point.

**Claim:** Schema validation catches syntax — not semantic errors like a wrong total.

In [10]:
def validate_invoice(data: dict) -> None:
    calculated = sum(item["amount_usd"] for item in data["line_items"])
    stated = data["stated_total_usd"]
    print(f"Line items sum: {calculated}")
    print(f"Stated total:   {stated}")
    if calculated != stated:
        raise ValueError(
            f"line_items sum to {calculated} but stated_total_usd is {stated}"
        )
    print("Validation passed — totals match.")

try:
    validate_invoice(extracted)
except ValueError as e:
    print(f"\nError: {e}")


Line items sum: 10500
Stated total:   11300

Error: line_items sum to 10500 but stated_total_usd is 11300


> **The schema is VALID. The data is WRONG.** Schemas kill syntax errors, not semantic errors.

**Claim:** Retry with error feedback fixes logic — it cannot invent facts missing from the source.

In [11]:
EXTRACT_INVOICE_TOOL_V2 = {
    **EXTRACT_INVOICE_TOOL,
    "input_schema": {
        **EXTRACT_INVOICE_TOOL["input_schema"],
        "properties": {
            **EXTRACT_INVOICE_TOOL["input_schema"]["properties"],
            "calculated_total_usd": {"type": "number"},
            "conflict_detected": {"type": "boolean"},
        },
        "required": EXTRACT_INVOICE_TOOL["input_schema"]["required"]
        + ["calculated_total_usd", "conflict_detected"],
    },
}

validation_error = "line_items sum to 10500 but stated_total_usd is 11300 — re-extract and add calculated_total_usd and a conflict_detected flag"

retry_resp = call_with_retry(
    model=MODEL,
    max_tokens=2048,
    tools=[EXTRACT_INVOICE_TOOL_V2],
    tool_choice={"type": "tool", "name": "extract_invoice"},
    messages=[
        {"role": "user", "content": f"Extract all invoice fields:\n\n{INVOICE_TEXT}"},
        {"role": "assistant", "content": extract_resp.content},
        {
            "role": "user",
            "content": [
                {
                    "type": "tool_result",
                    "tool_use_id": next(b.id for b in extract_resp.content if b.type == "tool_use"),
                    "content": json.dumps(extracted),
                },
                {"type": "text", "text": validation_error},
            ],
        },
    ],
)

retry_extracted = next(b.input for b in retry_resp.content if b.type == "tool_use")
print(json.dumps(retry_extracted, indent=2))

{
  "invoice_number": "INV-2026-0847",
  "vendor": "Meridian Consulting Ltd",
  "client": "Apex Retail Group",
  "invoice_date": "2026-08-12",
  "line_items": [
    {
      "description": "Strategy workshop facilitation",
      "amount_usd": 4500
    },
    {
      "description": "Market analysis report",
      "amount_usd": 3200
    },
    {
      "description": "Stakeholder interviews (12 sessions)",
      "amount_usd": 2800
    }
  ],
  "stated_total_usd": 11300,
  "calculated_total_usd": 10500,
  "conflict_detected": true,
  "payment_terms": "net_30"
}


> **Retries fix format/logic errors; they cannot conjure information absent from the source.** The conflict is flagged, not silently 'fixed'.

**Claim:** `tool_choice` mode controls whether Claude speaks, must call a tool, or must call a specific tool.

In [12]:
THANKS_MSG = "Thanks, that all looks right!"

modes = [
    ("auto", {"type": "auto"}),
    ("any", {"type": "any"}),
    ("forced extract_invoice", {"type": "tool", "name": "extract_invoice"}),
]

for label, choice in modes:
    print("\n" + "#" * 72)
    print(f"tool_choice = {label!r}")
    print("#" * 72)
    r = call_with_retry(
        model=MODEL,
        max_tokens=512,
        tools=[EXTRACT_INVOICE_TOOL],
        tool_choice=choice,
        messages=[{"role": "user", "content": THANKS_MSG}],
    )
    print(f"stop_reason: {r.stop_reason}")
    for block in r.content:
        if block.type == "text":
            print(f"text: {block.text[:200]}")
        elif block.type == "tool_use":
            print(f"tool: {block.name}({json.dumps(block.input)[:120]}...)")


########################################################################
tool_choice = 'auto'
########################################################################
stop_reason: end_turn
text: It looks like this might be the start of our conversation — I don't have any previous messages or invoice details from you yet!

Could you share the invoice you'd like me to process? You can paste the

########################################################################
tool_choice = 'any'
########################################################################
stop_reason: tool_use
tool: extract_invoice({"invoice_number": "<UNKNOWN>", "vendor": "<UNKNOWN>", "client": "<UNKNOWN>", "invoice_date": "<UNKNOWN>", "line_items":...)

########################################################################
tool_choice = 'forced extract_invoice'
########################################################################
stop_reason: tool_use
tool: extract_invoice({"invoice_number": "<UNKNOWN>", "vend

| `tool_choice` | When to use |
|---------------|-------------|
| `auto` (default) | General chat + optional tools — Claude decides |
| `any` | You need structured output every turn (audit pipelines) |
| `tool` + name | Extraction / classification with a fixed schema |

Use `auto` for assistants. Use forced tool for extraction. Use `any` sparingly — it forces a tool even when none fits.

## Section 4 — Prompt caching

### Same system prompt across hundreds of calls = production economics

In [13]:
SYSTEM_PROMPT = """NORTHWIND RETAIL — CUSTOMER SUPPORT POLICY MANUAL (v4.2)
Effective date: 1 January 2026 | Applies to: all customer-facing agents, chatbots, and escalation teams

=== 1. BRAND VOICE AND TONE ===
1.1 Always greet customers by first name when available. Use warm, professional language.
1.2 Never blame the customer for product issues, shipping delays, or billing errors.
1.3 Acknowledge frustration before offering solutions: "I understand how frustrating that must be."
1.4 Avoid jargon. Say "return label" not "RMA authorization document."
1.5 Do not use exclamation marks more than once per response.
1.6 Sign off with "— The Northwind Support Team" for email; chat may use shorter closings.

=== 2. REFUND AND RETURN POLICY ===
2.1 Standard return window: 30 calendar days from delivery date for most items.
2.2 Extended window: footwear and apparel may be returned within 45 days if unworn with tags attached.
2.3 Electronics return window: 14 days; must include all original accessories and packaging.
2.4 Final-sale items (marked "Final Sale" at checkout) are non-refundable and non-returnable.
2.5 Refunds are issued to the original payment method within 5–10 business days after warehouse receipt.
2.6 Store credit may be offered as an alternative; customer must explicitly accept.
2.7 Partial refunds apply when items are returned damaged due to customer misuse.
2.8 Shipping costs are non-refundable unless the return is due to Northwind error.
2.9 Gift purchases: refunds go to store credit issued to the recipient's email on file.
2.10 International orders: customer pays return shipping; customs fees are non-refundable.

=== 3. ESCALATION THRESHOLDS ===
3.1 Tier 1 (frontline): handles orders under $200 and standard returns.
3.2 Tier 2 (senior agent): required when order value is $200–$999 or customer requests supervisor.
3.3 Tier 3 (resolution specialist): required when order value exceeds $999 or legal threat is made.
3.4 Auto-escalate to Tier 2 if customer mentions "lawyer," "BBB," "chargeback," or "social media."
3.5 Refund requests over $500 require Tier 2 approval and fraud-screening check.
3.6 Escalation SLA: Tier 2 response within 4 hours; Tier 3 within 24 hours.
3.7 Document escalation reason in CRM ticket before transferring.

=== 4. SHIPPING AND DELIVERY ===
4.1 Standard shipping: 5–7 business days domestic.
4.2 Express shipping: 2–3 business days; no refund of shipping if customer changes mind.
4.3 Lost package: investigate after 7 days past expected delivery; reship or refund at agent discretion.
4.4 Damaged in transit: request photo within 48 hours; offer replacement or full refund including shipping.
4.5 Wrong item shipped: send correct item immediately; customer keeps or returns wrong item with prepaid label.

=== 5. BILLING AND PAYMENTS ===
5.1 Duplicate charges: verify in payment gateway; refund within 24 hours if confirmed.
5.2 Promo code issues: one courtesy re-application per customer per quarter.
5.3 Subscription cancellations: effective end of current billing period unless within 48 hours of charge.
5.4 Price-match requests: honor within 7 days of purchase with proof of competitor price.

=== 6. PRODUCT QUALITY AND WARRANTIES ===
6.1 Manufacturer warranty claims: direct customer to manufacturer after 30 days.
6.2 Northwind quality guarantee: defect within 90 days — replace or refund, no return shipping cost.
6.3 Food and perishables: no returns; quality issues require photo evidence for refund.

=== 7. PRIVACY AND DATA ===
7.1 Never share full payment card numbers; last four digits only.
7.2 Verify identity with order number + email or zip code before account changes.
7.3 GDPR/CCPA deletion requests: escalate to privacy team; do not process on chat.

=== 8. PROHIBITED ACTIONS ===
8.1 Never promise outcomes not covered by this policy.
8.2 Never offer cash equivalents (Venmo, wire) for refunds.
8.3 Never disparage competitors by name.
8.4 Never share internal discount codes not approved for public use.

=== 9. CHATBOT-SPECIFIC RULES ===
9.1 If confidence in answer is low, offer human handoff within 2 turns.
9.2 Do not process refunds over $100 without human review.
9.3 Log all policy exceptions with agent ID and justification.

=== 10. SEASONAL OVERRIDES ===
10.1 Holiday return extension (Nov 15 – Jan 15): purchases may be returned until Jan 31.
10.2 Black Friday/Cyber Monday: final-sale rules apply to doorbuster SKUs listed in promo sheet.

This document is the authoritative source for Northwind Retail customer support decisions.
Agents must cite the relevant section number when explaining policy to customers.
When policy conflicts arise, Tier 2+ agents determine the resolution and document the exception.
Repeat the customer's issue in your own words before citing policy sections.
For ambiguous cases, default to the option most favorable to the customer within policy bounds.
All timestamps in tickets must use UTC. Business hours for live chat: Mon–Fri 8am–10pm ET, Sat–Sun 9am–6pm ET.
After-hours tickets queue for next business day unless marked urgent (Tier 3 trigger).
Urgent queue target first response: 30 minutes during business hours.
Knowledge base articles must be updated within 48 hours of any policy change communicated here.
Training refresher required quarterly for all agents; certification quiz score minimum 85%.
Violations of sections 8.x may result in disciplinary action per HR handbook section 12.
Appendix A: SKU categories for extended return windows — footwear (FW-*), apparel (AP-*), electronics (EL-*).
Appendix B: Fraud indicators — multiple orders same card different addresses, velocity >3 orders/hour, high-risk BIN list.
Appendix C: Approved compensation caps — Tier 1: $25 store credit; Tier 2: $75; Tier 3: $200 without director approval.
End of Northwind Retail Customer Support Policy Manual v4.2.

=== 11. LOYALTY PROGRAM (NORTHSTAR REWARDS) ===
11.1 Points accrue at 1 point per $1 spent on eligible purchases.
11.2 Redemption: 100 points = $1 store credit; minimum redemption 500 points.
11.3 Points expire 24 months after earn date if account inactive.
11.4 Tier benefits: Silver ($0–499 annual spend), Gold ($500–1499), Platinum ($1500+).
11.5 Platinum members receive free express shipping on orders over $75.
11.6 Points are non-transferable except per written estate documentation.
11.7 Disputed point balances: freeze redemption until Tier 2 investigation completes.

=== 12. CORPORATE AND B2B ACCOUNTS ===
12.1 Net-30 terms available for approved accounts with credit application on file.
12.2 Purchase orders required for orders over $2,500 from B2B accounts.
12.3 Volume discount tiers: 5% at $10k annual, 10% at $50k, 15% at $100k — sales approval required.
12.4 Dedicated account manager assigned at $25k annual spend threshold.
12.5 B2B returns follow standard windows unless contract specifies otherwise.

=== 13. ACCESSIBILITY AND ACCOMMODATION ===
13.1 Offer phone callback for customers who cannot use chat due to disability.
13.2 Large-print invoices available on request within 2 business days.
13.3 Relay service calls accepted; do not end call if connection delay exceeds 30 seconds.

=== 14. SOCIAL MEDIA AND REPUTATION ===
14.1 Do not engage in arguments on public social channels.
14.2 Move public complaints to DM within one public reply offering help.
14.3 Escalate viral posts (>10k impressions mention) to communications team immediately.
14.4 Never share customer PII in any public response.

=== 15. METRICS AND QUALITY ASSURANCE ===
15.1 Target first-response time: chat 90 seconds, email 4 hours business time.
15.2 CSAT survey sent after every resolved ticket; target score 4.5/5.
15.3 QA samples 5% of tickets weekly; coaching required if score below 85%.
15.4 Repeat contact within 48 hours on same issue counts against FCR (first contact resolution).
15.5 Document resolution code from approved list — no free-text-only closures.

=== 16. INTERNATIONAL SUPPORT ===
16.1 Supported currencies for display: USD, CAD, GBP, EUR — charges settle in checkout currency.
16.2 VAT included in EU prices per local law; do not promise VAT refunds on chat without finance review.
16.3 APAC warehouse delays: set expectation of +3–5 days vs domestic during peak season.

=== 17. PRODUCT RECALLS ===
17.1 If SKU on active recall list, proactively offer full refund + return label regardless of purchase date.
17.2 Recall communications template must be used verbatim — legal pre-approved only.
17.3 Do not speculate on recall cause; cite official recall notice ID only.

=== 18. AGENT DESKTOP AND SECURITY ===
18.1 Lock workstation when away; auto-lock policy 5 minutes.
18.2 Two-factor authentication required for all CRM access.
18.3 USB storage devices prohibited on support floor workstations.
18.4 Customer screenshots may contain PII — store only in approved ticket attachment bucket.

Appendix D: Regional holiday closures — US, CA, UK calendars maintained in shared drive.
Appendix E: Approved vendor RMA partners and contact SLAs.
Appendix F: Sample escalation email templates (Tier 2 → Tier 3 handoff).
This manual supersedes all prior versions. Questions to policy-ops@northwind-retail.example.
"""

print(f"System prompt length: {len(SYSTEM_PROMPT):,} characters (~{len(SYSTEM_PROMPT)//4:,} tokens)")


System prompt length: 9,162 characters (~2,290 tokens)


**Claim:** First call with `cache_control` writes the system prompt to cache — watch `cache_creation_input_tokens`.

In [14]:
cache_call_1 = call_with_retry(
    model=MODEL,
    max_tokens=512,
    system=[
        {
            "type": "text",
            "text": SYSTEM_PROMPT,
            "cache_control": {"type": "ephemeral"},
        }
    ],
    messages=[
        {
            "role": "user",
            "content": "A customer wants to return shoes bought 45 days ago. What do I do?",
        }
    ],
)
show_response(cache_call_1)
usage_1 = cache_call_1.usage.model_dump()
print("\n>>> cache_creation_input_tokens:", usage_1.get("cache_creation_input_tokens", 0))

CONTENT BLOCKS

--- Block 1: type=text ---
## Returning Shoes Purchased 45 Days Ago

Here's how to handle this, step by step.

---

### Determine Eligibility First

Per **Section 2.2**, footwear (SKU category `FW-*`) has an **extended 45-day return window**, compared to the standard 30 days. A purchase made exactly 45 days ago falls **within this window** — but only if both conditions are met:

- ✅ **Unworn** — the shoes must not have been worn
- ✅ **Tags still attached**

---

### Check for Holiday Override

Per **Section 10.1**, if the original purchase was made between **November 15 and January 15**, the return deadline may extend to **January 31** — which could give the customer additional time even if the 45-day window has technically closed.

---

### If Eligible — Process the Return

1. Verify order number + email or zip code to confirm identity (**Section 7.2**)
2. Confirm the shoes are unworn with tags attached
3. Issue a return label (standard return shipping is customer-resp

USAGE
{
  "cache_creation": {
    "ephemeral_1h_input_tokens": 0,
    "ephemeral_5m_input_tokens": 2601
  },
  "cache_creation_input_tokens": 2601,
  "cache_read_input_tokens": 0,
  "inference_geo": "global",
  "input_tokens": 23,
  "output_tokens": 476,
  "output_tokens_details": null,
  "server_tool_use": null,
  "service_tier": "standard"
}

>>> cache_creation_input_tokens: 2601


**Claim:** Second call with the same cached system prompt reads from cache — `cache_read_input_tokens` > 0, `input_tokens` tiny.

In [15]:
cache_call_2 = call_with_retry(
    model=MODEL,
    max_tokens=512,
    system=[
        {
            "type": "text",
            "text": SYSTEM_PROMPT,
            "cache_control": {"type": "ephemeral"},
        }
    ],
    messages=[
        {
            "role": "user",
            "content": "What's the escalation threshold for refunds?",
        }
    ],
)
show_response(cache_call_2)
usage_2 = cache_call_2.usage.model_dump()
print("\n>>> cache_read_input_tokens:", usage_2.get("cache_read_input_tokens", 0))
print(">>> input_tokens:", usage_2.get("input_tokens", 0))

CONTENT BLOCKS

--- Block 1: type=text ---
Based on the Northwind Retail Customer Support Policy Manual, here's a summary of the escalation thresholds related to refunds:

## Escalation Thresholds for Refunds

| Trigger | Required Action |
|---|---|
| Refund on order **under $200** | Tier 1 (frontline) can handle |
| Refund on order **$200–$999** | Requires **Tier 2** (senior agent) |
| Refund on order **over $999** | Requires **Tier 3** (resolution specialist) |
| Refund request **over $500** | Requires Tier 2 approval + fraud-screening check |
| Chatbot handling refund **over $100** | Requires human review before processing |

### Additional Triggers That Force Escalation (§3.4)
Even on lower-value orders, auto-escalation to **Tier 2** is required if the customer mentions:
- "Lawyer" or legal threats
- BBB (Better Business Bureau)
- Chargeback
- Social media

### Response SLAs (§3.6)
- **Tier 2:** must respond within **4 hours**
- **Tier 3:** must respond within **24 hours**

> **Rel

USAGE
{
  "cache_creation": {
    "ephemeral_1h_input_tokens": 0,
    "ephemeral_5m_input_tokens": 0
  },
  "cache_creation_input_tokens": 0,
  "cache_read_input_tokens": 2601,
  "inference_geo": "global",
  "input_tokens": 15,
  "output_tokens": 332,
  "output_tokens_details": null,
  "server_tool_use": null,
  "service_tier": "standard"
}

>>> cache_read_input_tokens: 2601
>>> input_tokens: 15


**Side-by-side usage:**

| | Call 1 (cache write) | Call 2 (cache read) |
|---|---------------------|---------------------|
| `input_tokens` | (see output above) | (see output above) |
| `cache_creation_input_tokens` | > 0 on first call | 0 |
| `cache_read_input_tokens` | 0 | > 0 on second call |

**Cache reads are ~90% cheaper.** Same system prompt across hundreds of agent calls = this is the whole economics of production agents.

*(Exam: know it exists. Production: know it cold.)*

## Section 5 — Wrap-up

| Tonight's section | Assignment 1 pass criterion |
|-------------------|----------------------------|
| Section 1 — `stop_reason` + `max_tokens` | #1 Handle `max_tokens` explicitly in your agent loop |
| Section 2 — tool loop by hand | #2 Branch on `stop_reason`, not iteration count |
| Section 2 — structured tool errors | #3 Return structured errors from tools (`error`, `errorCategory`, `isRetryable`) |
| Section 3 — schema + validation + retry | #4 Validate extracted data; retry with error feedback |
| Section 4 — prompt caching | #5 Demonstrate cache write then cache read in usage stats |

**Streaming demo:** Notebooks buffer output — run `stream_demo.py` from the terminal for the live streaming segment.

```bash
python stream_demo.py "Explain in 3 sentences why streaming improves perceived latency."
```

**Resources (placeholders):**
- Starter repo: `https://github.com/example/claude-architect-bootcamp-starter`
- Community channel: `#claude-architect-bootcamp`